# Cost Optimization and Telemetry-Based Scaling
Companion notebook for the **Cost Optimization and Telemetry-Based Scaling** section of Chapter 12.

Measure cost at the level where decisions happen: **per request, per user, and per workflow**. This is the book's telemetry record, unchanged:

In [1]:
telemetry_record = {
    "request_id": "req_4821",
    "user_id": "u_19",
    "workflow": "claims_review",
    "model": "large-reasoner",
    "prompt_tokens": 8200,
    "completion_tokens": 1400,
    "tool_calls": 6,
    "retries": 2,
    "latency_ms": 18400,
    "estimated_cost_usd": 0.41,
}
print(telemetry_record)

{'request_id': 'req_4821', 'user_id': 'u_19', 'workflow': 'claims_review', 'model': 'large-reasoner', 'prompt_tokens': 8200, 'completion_tokens': 1400, 'tool_calls': 6, 'retries': 2, 'latency_ms': 18400, 'estimated_cost_usd': 0.41}


This notebook loads the real `budget-controls.yaml` shipped alongside it -- the book's cost caps, unchanged:

In [2]:
import yaml

budget = yaml.safe_load(open("budget-controls.yaml"))["budget_controls"]
print(budget)

{'max_cost_per_run_usd': 0.5, 'max_tool_calls': 8, 'max_retries': 2, 'max_tokens': 12000, 'on_limit_hit': 'fallback_to_small_model'}


The config alone doesn't enforce anything -- here's the check a run would actually go through, using the shipped `telemetry_record` above and one that exceeds the caps:

In [3]:
def check_budget(record: dict, budget: dict) -> dict:
    total_tokens = record["prompt_tokens"] + record["completion_tokens"]
    breaches = []
    if record["estimated_cost_usd"] > budget["max_cost_per_run_usd"]:
        breaches.append("cost")
    if record["tool_calls"] > budget["max_tool_calls"]:
        breaches.append("tool_calls")
    if record["retries"] > budget["max_retries"]:
        breaches.append("retries")
    if total_tokens > budget["max_tokens"]:
        breaches.append("tokens")
    if breaches:
        return {"action": budget["on_limit_hit"], "breached": breaches}
    return {"action": "continue", "breached": []}


print(check_budget(telemetry_record, budget))

expensive_record = {**telemetry_record, "estimated_cost_usd": 0.85, "tool_calls": 11}
print(check_budget(expensive_record, budget))

{'action': 'continue', 'breached': []}
{'action': 'fallback_to_small_model', 'breached': ['cost', 'tool_calls']}


The first record (retries=2, right at the `max_retries` cap, but nothing *over* it) continues normally. The second breaches both cost and tool-call limits and downgrades to `fallback_to_small_model` -- exactly the behavior the book describes: "when a run crosses a threshold, downgrade the path."

**Model routing**: send simple work to a small model, reserve the large one for hard reasoning or high-value users:

In [4]:
def choose_model(task_difficulty: str, user_tier: str) -> str:
    if task_difficulty == "hard" or user_tier == "enterprise":
        return "large-reasoner"
    if task_difficulty == "medium":
        return "mid-tier-model"
    return "small-fast-model"


print(choose_model("easy", "standard"))
print(choose_model("hard", "standard"))
print(choose_model("easy", "enterprise"))  # high-value user still gets the stronger model

small-fast-model
large-reasoner
large-reasoner


Scale from **business value**, not just raw request count -- a queue of low-value background summaries shouldn't push out revenue-critical support or fraud-review traffic. A tiny priority-ordering example over the same telemetry shape:

In [5]:
requests = [
    {"workflow": "background_summary", "priority_weight": 1},
    {"workflow": "fraud_review", "priority_weight": 10},
    {"workflow": "support_ticket", "priority_weight": 5},
]
ordered = sorted(requests, key=lambda r: r["priority_weight"], reverse=True)
print([r["workflow"] for r in ordered])

['fraud_review', 'support_ticket', 'background_summary']
